# Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Load data 
raw_data = pd.read_csv("Dataset/kirana.csv")
raw_data["timestamp"] = pd.to_datetime(raw_data["timestamp"])
raw_data = raw_data.set_index('timestamp').sort_index()

raw_data.head(5)

In [ ]:
# Preparing Cash data

data = raw_data[raw_data['payment_method']=='cash'].resample('h').size().to_frame('count')

data = data.asfreq('h',fill_value = 0)

data.head(6)

# Feature Engineering

In [ ]:
data['lag_24'] = data['count'].shift(24)
data['hour'] = data.index.hour
data['dayofweek'] = data.index.dayofweek
data = data.dropna()

data

In [ ]:
#Split data for testing and training
split = int(len(data) * 0.8)
train = data.iloc[:split]
test = data.iloc[split:].copy()

# Model Training

In [ ]:
model = xgb.XGBRegressor(n_estimators=100, random_state=42)
model.fit(train[['hour', 'dayofweek', 'lag_24']], train['count'])

In [ ]:
test['pred_count'] = model.predict(test[['hour', 'dayofweek', 'lag_24']])

In [ ]:
# 6. Apply "Daily Average" Logic
# Calculate the daily average for ACTUAL data (The Truth)
daily_avg_actual = test.groupby(test.index.date)['count'].transform('mean')

# Calculate the daily average for PREDICTED data (The Model's guess)
daily_avg_pred = test.groupby(test.index.date)['pred_count'].transform('mean')

# Define "Busy" as being above that day's average
test['busy_actual'] = (test['count'] > daily_avg_actual).astype(int)
test['busy_predicted'] = (test['pred_count'] > daily_avg_pred).astype(int)

# Evaluation

In [ ]:
cm = confusion_matrix(test['busy_actual'], test['busy_predicted'])
precision = precision_score(test['busy_actual'], test['busy_predicted'])
recall = recall_score(test['busy_actual'], test['busy_predicted'])

print("--- Evaluation Results ---")
print(f"Precision (Trustworthiness): {precision:.2%}")
print(f"Recall (Hit Rate): {recall:.2%}")
print("\nConfusion Matrix:")
print(cm)
print("\n[True Negatives  False Alarms]")
print("[Missed Spikes   True Spikes ]")

In [ ]:
# 1. Select only the columns we care about
results_table = test[['count', 'pred_count', 'busy_actual', 'busy_predicted']].copy()

# 2. Rename them to be human-readable
results_table.columns = [
    'Actual Cash Txns', 
    'Predicted Cash Txns', 
    'Is Actual Busy Hour?', 
    'Is Predicted Busy Hour?'
]

# 3. Formatting: Round the predicted numbers to 1 decimal place
results_table['Predicted Cash Txns'] = results_table['Predicted Cash Txns'].round(1)

# 4. Map the "1" and "0" to "Yes" and "No" for easier reading
results_table['Is Actual Busy Hour?'] = results_table['Is Actual Busy Hour?'].map({1: 'YES', 0: '-'})
results_table['Is Predicted Busy Hour?'] = results_table['Is Predicted Busy Hour?'].map({1: 'YES', 0: '-'})

# --- Display the Table ---

# Option A: Print the last 24 hours of data to the screen
print("\n--- Prediction Results (Last 24 Hours) ---")
results_table.tail(24)

In [ ]:
import joblib
# 'model' is the variable name of your XGBRegressor
joblib.dump(model, 'model_spikes.pkl')